In [7]:
import pyspark
import pandas as pd
import wget
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
### Question 1: Execute spark.version


spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 4.1.1


In [3]:
### Question 2: Yellow November 2025


# download file
!wget --no-check-certificate https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

file_path = "yellow_tripdata_2025-11.parquet"
size_mb = os.path.getsize(file_path) / (1024 * 1024)
print(f"File size: {size_mb:.2f} MB")

File size: 67.84 MB


--2026-03-10 00:16:25--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.38.83, 18.239.38.147, 18.239.38.163, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.38.83|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: 'yellow_tripdata_2025-11.parquet'

     0K .......... .......... .......... .......... ..........  0% 87.5K 13m13s
    50K .......... .......... .......... .......... ..........  0%  216K 9m17s
   100K .......... .......... .......... .......... ..........  0%  206K 8m3s
   150K .......... .......... .......... .......... ..........  0%  642K 6m29s
   200K .......... .......... .......... .......... ..........  0%  783K 5m29s
   250K .......... .......... .......... .......... ..........  0%  245K 5m21s
   300K .......... .......... .......... .......... 

In [4]:
# read parquet
taxi = spark.read.parquet("yellow_tripdata_2025-11.parquet")

# repartition
taxi = taxi.repartition(4)

# write parquet
taxi.write.mode("overwrite").parquet("yellow")

In [5]:
# Question 3: Count records


taxi.filter((taxi.tpep_pickup_datetime >= '2025-11-15') & (taxi.tpep_pickup_datetime < '2025-11-16')).count()

162604

In [8]:
# Question 4: Longest trip in dataset


taxi.withColumn('trip_duration', (F.unix_timestamp(taxi.tpep_dropoff_datetime) - F.unix_timestamp(taxi.tpep_pickup_datetime)) / 3600).\
    select('trip_duration').sort('trip_duration', ascending = False).show(1)

+-----------------+
|    trip_duration|
+-----------------+
|90.64666666666666|
+-----------------+
only showing top 1 row


In [9]:
# Question 5: User Interface > or we can forward 4040 port


spark._activeSession

In [13]:
# Question 6: Least frequent pickup location zone

# !wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
zone = spark.read.csv('taxI_zone_lookup.csv', header = True, inferSchema = True)
merged = taxi.join(zone, zone.LocationID == taxi.PULocationID, 'left').\
                withColumnRenamed('Zone', 'pickup_zone').drop('Borough', 'service_zone', 'LocationID')
merged.groupby('pickup_zone').count().sort('count', ascending = True).head(5)

[Row(pickup_zone="Governor's Island/Ellis Island/Liberty Island", count=1),
 Row(pickup_zone='Arden Heights', count=1),
 Row(pickup_zone="Eltingville/Annadale/Prince's Bay", count=1),
 Row(pickup_zone='Port Richmond', count=3),
 Row(pickup_zone='Rikers Island', count=4)]